# MLB Statcast 스트라이크 예측 데이터셋 구축

`pybaseball_statcast_raw_sequence_strike_target.md`의 설계를 실행 가능한 파이프라인으로 옮긴 노트북입니다.

- 원본은 월별 Parquet + JSON 메타데이터로 보존합니다.
- 정규시즌(`game_type == 'R'`)과 유효한 `type`만 모델 데이터에 사용합니다.
- 타깃은 `type == 'S'`인 `is_strike`입니다.
- 모든 이력/rolling 피처는 현재 투구를 제외하도록 `shift(1)`을 적용합니다.
- 현재 투구의 결과, 위치, 물리 로그, WPA, 타구 결과는 모델 입력에서 제거합니다.
- 학습은 2017–2018년, 테스트는 2019년으로 시간 분할합니다.

> 전체 3개 시즌은 다운로드와 피처 생성에 시간이 오래 걸리고 메모리를 많이 사용합니다. 먼저 아래 `QUICK_TEST=True`, `RUN_DOWNLOAD=True`로 짧은 구간을 검증한 뒤 전체 실행으로 전환하세요.

In [ ]:
# 최초 1회만 실행
%pip install -q pybaseball pandas numpy pyarrow

In [ ]:
from pathlib import Path
import pandas as pd

from build_statcast_strike_dataset import (
    BuildConfig, RAW_COLUMNS, assert_integrity, audit_raw, build_features, collect_raw,
    integrity_report, load_raw, make_model_dataset, save_dataset,
)

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 100)

## 1. 실행 설정

처음에는 빠른 테스트가 권장됩니다. 전체 구축 시 `QUICK_TEST=False`, 원본이 없으면 `RUN_DOWNLOAD=True`로 바꾸세요. 이미 저장된 월별 파일은 자동으로 건너뜁니다.

In [ ]:
QUICK_TEST = True
RUN_DOWNLOAD = False  # 원본 파일이 없으면 True

config = BuildConfig()
if QUICK_TEST:
    config.seasons = {2019: ('2019-03-20', '2019-03-27')}
    config.raw_dir = Path('data/statcast_raw_sequence_quick_test')
    config.processed_dir = Path('data/processed_quick_test')

config

## 2. 원본 수집

수집 단계에서는 필터링하거나 중복을 제거하지 않습니다. pybaseball 반환 행과 열을 그대로 저장하고, 원래 반환 순서인 `_source_row`와 청크 내 정렬 순서인 `_sequence_in_chunk`를 추가합니다.

In [ ]:
if RUN_DOWNLOAD:
    manifest = collect_raw(config)
    display(manifest)
else:
    print('다운로드를 생략했습니다. 기존 원본 Parquet를 사용합니다.')

## 3. 원본 로드 및 감사

필요한 열만 읽어 메모리 사용량을 줄입니다. 시즌별 스키마 차이로 일부 열이 없더라도 로더가 처리합니다.

In [ ]:
raw = load_raw(config, columns=RAW_COLUMNS)
display(audit_raw(raw))
print('기간:', raw['game_date'].min(), '~', raw['game_date'].max())
display(raw['game_type'].value_counts(dropna=False).rename_axis('game_type').to_frame('rows'))

## 4. 누수 방지 피처 생성

상황 피처, 이전 투구 1–3개, 최근 20구 사용률/평균 물리값, 과거 스트라이크율, 이전 경기 투구 수, 휴식일, 과거 정보만 이용한 LI를 생성합니다. VAA/HAA는 현재 투구값 자체가 아니라 직전 투구값만 남깁니다.

In [ ]:
enriched = build_features(raw, config)
model_df = make_model_dataset(enriched, config)
print('모델 데이터 크기:', model_df.shape)
display(model_df.head())

## 5. 검증

중복 키, 타깃, 시간 분할, 대표 누수 열을 확인합니다. 빠른 테스트는 2019년만 사용하므로 모든 행이 `test`인 것이 정상입니다.

In [ ]:
report = integrity_report(model_df, config, model_dataset=True)
display(report.style.applymap(
    lambda value: 'font-weight: bold; color: ' + ({'PASS': 'green', 'WARN': 'orange', 'FAIL': 'red'}.get(value, 'black'))
))
assert_integrity(report)

summary = model_df.groupby('split', observed=True).agg(
    rows=('is_strike', 'size'),
    strike_rate=('is_strike', 'mean'),
    first_date=('game_date', 'min'),
    last_date=('game_date', 'max'),
)
display(summary)
print('검증 통과')

## 6. 저장

전체 모델 데이터, 학습/테스트 분할 파일, 스키마 JSON을 각각 저장합니다.

In [ ]:
outputs = save_dataset(model_df, config)
for name, path in outputs.items():
    print(f'{name:>6}: {path.resolve()}')

## 전체 시즌 실행 방법

1. 설정 셀에서 `QUICK_TEST = False`로 바꿉니다.
2. 최초 수집이면 `RUN_DOWNLOAD = True`로 바꿉니다.
3. 수집부터 저장까지 순서대로 실행합니다.
4. 다운로드 완료 후 재실행할 때는 `RUN_DOWNLOAD = False`로 두면 저장된 원본을 재사용합니다.

현재 구종을 알고 난 뒤 스트라이크 여부를 예측한다는 가정이 기본입니다. 구종 선택 전 예측이 목적이면 `config.include_current_pitch_type = False`로 설정하세요.